In [2]:
#!/usr/bin/env python3
"""
Split the 3-panel figure into three separate figures (one per test).

Reads: all.tsv (tab-separated) with columns:
  - 'ags_number', 'noise', 'method' (or 'run_id'), and roc columns:
  - 'roc_aucs_test', 'roc_aucs_testAB', 'roc_aucs_testAG' (or similar)

Outputs:
  - auc_by_method_colored_testAB.png
  - auc_by_method_colored_testAG.png
  - auc_by_method_colored_test.png
"""
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import cycle

IN_TSV = "temp/all.tsv"
OUT_BASENAME = "auc_by_method_colored"
FIGSIZE_SINGLE = (6, 5)  # per-figure size

# user-specified methods and palette (kept order-to-color mapping)
methods_lst = ['grad01_avg', 'grad01_max', 'gradient_conf_avg',
               'gradient_conf_max', 'gradient_input_avg', 'gradient_input_max',
               'model_grad_avg',
               'model_grad_max', 'qbc', 'threshold_avg',
               'threshold_max', 'feature_eucl', 'random']

base_palette = ["#004949", "#009292", "#ff6db6",
                "#ffb6db", "#490092", "#006ddb",
                "#b6dbff",
                "#920000", "#924900", "#db6d00",
                "#24ff24", "#ffa500", "#ffff6d"]

# integrator (prefer trapezoid)
integrate = getattr(np, "trapezoid", np.trapz)

# Read data
df = pd.read_csv(IN_TSV, sep='\t')
df.columns = [c.strip() for c in df.columns]

# Make sure we have a method label
if 'method' in df.columns:
    df['method'] = df['method'].astype(str)
elif 'run_id' in df.columns:
    df['method'] = df['run_id'].astype(str)
else:
    df['method'] = 'method_unknown'

# Ensure numeric types
df['noise'] = pd.to_numeric(df['noise'], errors='coerce')
df['ags_number'] = pd.to_numeric(df['ags_number'], errors='coerce')

# Detect roc columns for the three tests
def find_roc_col(test_name):
    cand = f"roc_aucs_{test_name}" if test_name != 'test' else "roc_aucs_test"
    if cand in df.columns:
        return cand
    for c in df.columns:
        if 'roc' in c.lower() and test_name.lower() in c.lower():
            return c
    for c in df.columns:
        if c.lower().startswith('roc_aucs'):
            return c
    raise KeyError(f"No ROC column found for test '{test_name}'")

tests = ['testAB', 'testAG', 'test']
roc_cols = {t: find_roc_col(t) for t in tests}

# Compute integrated AUC per (test, noise, method)
rows = []
for test in tests:
    col = roc_cols[test]
    for noise_val, df_noise in df.groupby('noise'):
        for method_name, df_m in df_noise.groupby('method'):
            mean_by_ags = df_m.groupby('ags_number')[col].mean()
            if mean_by_ags.dropna().shape[0] < 2:
                continue
            mean_by_ags.index = pd.to_numeric(mean_by_ags.index, errors='coerce')
            mean_by_ags = mean_by_ags.dropna().sort_index()
            x = mean_by_ags.index.values.astype(float)
            y = mean_by_ags.values.astype(float)
            auc = integrate(y, x)
            x_range = x.max() - x.min() if x.size > 0 else 0.0
            auc_norm = auc / x_range if x_range > 0 else np.nan
            rows.append({
                'test': test,
                'noise': float(noise_val),
                'method': method_name,
                'auc': float(auc),
                'auc_norm': float(auc_norm) if not np.isnan(auc_norm) else np.nan,
                'x_min': float(x.min()),
                'x_max': float(x.max()),
                'n_points': int(x.size)
            })

if not rows:
    raise RuntimeError("No AUC rows computed — check 'all.tsv' content and columns.")

df_auc = pd.DataFrame(rows)

# Choose which column to plot: 'auc' (integrated) or 'auc_norm' (normalized)
PLOT_NORMALIZED = False
plot_col = 'auc_norm' if PLOT_NORMALIZED else 'auc'

# Prepare color mapping
palette_map = {m: c for m, c in zip(methods_lst, base_palette)}
fallback_colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', [])
fallback_cycle = cycle(fallback_colors)

present_methods = sorted(df_auc['method'].unique())
method_colors = {}
for m in present_methods:
    if m in palette_map:
        method_colors[m] = palette_map[m]
    else:
        method_colors[m] = next(fallback_cycle)

# markers
markers = ['o', 's', 'D', '^', 'v', '<', '>', 'P', 'X', '*']
marker_cycle = cycle(markers)
method_markers = {m: next(marker_cycle) for m in present_methods}

# compute a global y-limits so all three figures share the same vertical span
ymin = df_auc[plot_col].min()
ymax = df_auc[plot_col].max()
yrange = (ymax - ymin) if (pd.notna(ymax) and pd.notna(ymin)) else 1.0
y_pad = max(0.5, 0.06 * yrange)
global_ylim = (ymin - y_pad, ymax + y_pad)

# title map (kept from your version)
dict0 = {'testAB':'TestSharedAG', 'testAG':'TestSharedAB', 'test':'Test'}

os.makedirs(".", exist_ok=True)

# Create and save separate figures
for test in tests:
    col = roc_cols[test]
    df_t = df_auc[df_auc['test'] == test].copy()
    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE_SINGLE)

    ax.set_title(f"{dict0.get(test, test)}", fontsize=20, pad=12)
    ax.set_xlabel("noise")
    ax.set_frame_on(False)
    ax.yaxis.grid(False)

    if df_t.empty:
        ax.text(0.5, 0.5, "no data", ha='center', va='center', fontsize=12)
    else:
        noises_sorted = sorted(df_t['noise'].unique())
        for method_name in methods_lst + [m for m in present_methods if m not in methods_lst]:
            df_m = df_t[df_t['method'] == method_name].set_index('noise').reindex(noises_sorted)
            if df_m.empty or df_m[plot_col].dropna().shape[0] < 2:
                continue
            x = np.array(noises_sorted)
            y = df_m[plot_col].values.astype(float)
            color = method_colors.get(method_name)
            marker = method_markers.get(method_name, 'o')
            linestyle = '-' if method_name.lower() != 'random' else '--'
            ax.plot(x, y, label=method_name, marker=marker, linestyle=linestyle,
                    linewidth=1.6, markersize=6, color=color)

        ax.set_xticks(noises_sorted)
        ax.set_xticklabels([f"{v:.2f}".rstrip('0').rstrip('.') for v in noises_sorted], rotation=45)
        ax.set_ylim(global_ylim)

    # legend to the right of each single figure
    handles, labels = ax.get_legend_handles_labels()
    ordered = []
    ordered_labels = []
    for m in methods_lst + [m for m in present_methods if m not in methods_lst]:
        if m in labels:
            idx = labels.index(m)
            ordered.append(handles[idx])
            ordered_labels.append(labels[idx])
    for h, l in zip(handles, labels):
        if l not in ordered_labels:
            ordered.append(h); ordered_labels.append(l)

    fig.legend(ordered, ordered_labels, loc='center right', bbox_to_anchor=(1.25, 0.5),
               fontsize=9, frameon=True)
    fig.subplots_adjust(right=0.78)  # make room for legend
    ax.set_ylabel("AUC (normalized)" if PLOT_NORMALIZED else "AUC")

    out_png = f"../results/graphs/{OUT_BASENAME}_{test}.png"
    plt.tight_layout()
    fig.savefig(out_png, dpi=1000, bbox_inches='tight')
    print(f"Saved figure for {test} -> {out_png}")
    plt.close(fig)


Saved figure for testAB -> ../results/graphs/auc_by_method_colored_testAB.png
Saved figure for testAG -> ../results/graphs/auc_by_method_colored_testAG.png
Saved figure for test -> ../results/graphs/auc_by_method_colored_test.png
